In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%pip install scikit-learn

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import numpy as np

In [ ]:
API_KEY = "c1de3d9b-e0a5-4146-a7fd-0e68ed812dc0" 
base_URL = "https://api.pricempire.com/v4/paid/items/prices"
headers = {"Authorization": f"Bearer {API_KEY}"}

In [ ]:
def fetch_prices():   
    params = {
        "sources": "buff163,dmarket,skinport", 
        "currency": "USD",
        "avg": "true",
        "median": "true"
    }
    response = requests.get(base_URL, headers=headers, params=params)
    
    if response.status_code != 200:
        raise Exception(f"API request failed with status code {response.status_code}: {response.text}")
    
    return response.json()

def fetch_price_history():
    history_URL = f"{base_URL}/history"
    params = {
        "app_id": 730,
        "provider_key": "buff163",
        "currency": "USD",
        "from_date": "2025-08-01",
        "to_date": "2025-10-29"
    }

    response = requests.get(history_URL, headers=headers, params=params)

    if response.status_code != 200:
        raise Exception(f"API request failed with status code {response.status_code}: {response.text}")
    
    return response.json()

In [ ]:
data = fetch_price_history()

records = []
price_data = data["data"] 

for name, history in price_data.items():
    for timestamp, price in history.items():
        records.append((name, timestamp, price))

records_np = np.array(records, dtype=object)

np.set_printoptions(linewidth=150, suppress=True)
print("Fetched historical data (name, timestamp, price):\n")
print(records_np)

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(records_np, columns=["market_hash_name", "timestamp", "price"])
df["timestamp"] = df["timestamp"].astype(int)
df["price"] = df["price"].astype(float)//100
df["date"] = pd.to_datetime(df["timestamp"], unit="s")
df = df.sort_values(by="date")

skin_names = df["market_hash_name"].unique().tolist()
skin_names = sorted(skin_names)

output_path = "skin_list.txt"
with open(output_path, "w", encoding="utf-8") as f:
    for name in skin_names:
        f.write(name + "\n")

print(f"✅ Saved {len(skin_names)} unique skins to {output_path}")


df["price_usd"] = df["price"] 

unique_skins = ["Autograph Capsule | Counter Logic Gaming | Cologne 2016"]

plt.figure(figsize=(14, 8))
for name in unique_skins:
    subset = df[df["market_hash_name"] == name]
    plt.plot(subset["date"], subset["price_usd"], label=name)

plt.title("Skin Price History", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend(
    fontsize=8,
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    title="Skins"
)
plt.grid(True)
plt.tight_layout(rect=[0, 0, 0.8, 1])
plt.show()


In [ ]:
df["pct_change"] = df.groupby("market_hash_name")["price_usd"].pct_change()
df["rolling_mean"] = df.groupby("market_hash_name")["price_usd"].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean())
df["deviation_from_mean"] = (df["price_usd"] - df["rolling_mean"]) / df["rolling_mean"]
df["spike_flag"] = (
    (abs(df["pct_change"]) > 0.5)|        
    (abs(df["deviation_from_mean"]) > 0.7)
)
#Searching for the spike of a particular skin
for row in df[df["spike_flag"]].itertuples():
    if row.market_hash_name == "USP-S | Black Lotus (Factory New)":
        print(f"Spike detected for {row.market_hash_name} on {row.date.date()}:")
        print(f"  Price: ${row.price_usd:.2f}, Pct Change: {row.pct_change:.2%}, Deviation from Mean: {row.deviation_from_mean:.2%}")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


upper_limit_usd = 500  
filtered_df = df[df["price"] < upper_limit_usd]

print(f"Found {filtered_df['market_hash_name'].nunique()} skins below ${upper_limit_usd}.")

df["market_hash_name"] = df["market_hash_name"].str.strip()

unique_skins = [
    "AK-47 | Redline (Field-Tested)",
    "SSG 08 | Ghost Crusader (Factory New)",
    "USP-S | Black Lotus (Factory New)",
    "AUG | Arctic Wolf (Factory New)",
    "AK-47 | Safari Mesh (Factory New)",
    "Sticker | Evil Geniuses (Holo) | Stockholm 2021"
]

plt.figure(figsize=(14, 8))
for name in unique_skins:
    subset = filtered_df[filtered_df["market_hash_name"] == name]
    plt.plot(subset["date"], subset["price"], label=name)

plt.title(f"Skin Price History (< ${upper_limit_usd} USD)", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend(
    fontsize=8,
    loc='center left',
    bbox_to_anchor=(1, 0.5),
    title="Skins"
)
plt.grid(True)
plt.tight_layout(rect=[0, 0, 0.8, 1])
plt.show()


In [ ]:
feature_df = df.groupby("market_hash_name").agg({
    "price_usd": "mean",
    "pct_change": ["mean", "std"],
    "deviation_from_mean": "mean",
    "spike_flag": "sum"
})

feature_df.columns = ['_'.join(col).strip() for col in feature_df.columns.values]

feature_df = feature_df.rename(columns={
    "price_usd_mean": "mean_price_usd",
    "pct_change_mean": "mean_pct_change",
    "pct_change_std": "volatility",
    "deviation_from_mean_mean": "mean_deviation",
    "spike_flag_sum": "spike_count"
})

In [ ]:
# Linear Predictor
def train_risk_model(feature_df):
    """
    feature_df: per-skin aggregated features.
    Must contain: mean_price_usd, mean_pct_change, volatility, mean_deviation, spike_count
    Returns: trained LDA model, scaler, and the cleaned feature_df with risky_label.
    """

    # --- 0. Work on a copy so we don't mutate the caller's dataframe ---
    feature_df = feature_df.copy()

    # --- 1. Create pseudo-label (risky_label) based on extremes ---
    spike_threshold = feature_df["spike_count"].quantile(0.9)  # top 10% spikes
    vol_threshold   = feature_df["volatility"].quantile(0.9)   # top 10% volatility

    feature_df["risky_label"] = np.where(
        (feature_df["spike_count"] >= spike_threshold) |
        (feature_df["volatility"]   >= vol_threshold),
        1,
        0
    )

    # --- 2. Build X matrix from the selected features ---
    X = feature_df[[
        "mean_price_usd",
        "mean_pct_change",
        "volatility",
        "mean_deviation",
        "spike_count"
    ]].copy()

    y = feature_df["risky_label"].values

    # --- 3. Clean X: handle inf, NaN, giant values ---

    # Replace +/- inf with NaN
    X = X.replace([np.inf, -np.inf], np.nan)

    # Fill NaN with 0 (you could also choose column medians if you prefer)
    X = X.fillna(0)

    # Clip insane outliers so scaling doesn't explode.
    # This makes sure a single crazy number (like 1e12) doesn't dominate MinMax.
    X = np.clip(X, -1e6, 1e6)

    # At this point X is a NumPy array again
    X = np.array(X, dtype=float)

    # --- 4. Train/test split ---
    # It's possible all risky_label are 0 or 1 if data is super uniform,
    # so we wrap stratify in a try/except for robustness.
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=0, stratify=y
        )
    except ValueError:
        # stratify can fail if there's only one class. fallback:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=0
        )

    # --- 5. Scale features 0-1 ---
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # --- 6. Train LDA ---
    lda = LinearDiscriminantAnalysis()
    lda.fit(X_train_scaled, y_train)

    # --- 7. Evaluate ---
    train_acc = lda.score(X_train_scaled, y_train)
    test_acc  = lda.score(X_test_scaled,  y_test)

    print(f"[LDA] Training accuracy: {train_acc:.3f}")
    print(f"[LDA] Testing  accuracy: {test_acc:.3f}")
    print("Class balance (0 normal / 1 risky):")
    print(pd.Series(y).value_counts())

    return lda, scaler, feature_df


In [ ]:
def generate_client_report(feature_df, lda, scaler, alpha=0.25):
    """
    feature_df: same shape/features as in train_risk_model()
    lda, scaler: from train_risk_model()
    alpha: how aggressively we discount high-risk skins (0.25 => up to 25%)
    Returns: report_df with all pricing info
    """

    # Select same features used for training
    X_all = feature_df[[
        "mean_price_usd",
        "mean_pct_change",
        "volatility",
        "mean_deviation",
        "spike_count"
    ]].copy()

    # --- 🧹 Clean data before scaling ---
    import numpy as np
    X_all = X_all.replace([np.inf, -np.inf], np.nan)
    X_all = X_all.fillna(0)
    X_all = np.clip(X_all, -1e6, 1e6) 
    X_all = np.array(X_all, dtype=float)

    # --- Scale features ---
    X_all_scaled = scaler.transform(X_all)

    # --- Predict risk probability ---
    if hasattr(lda, "predict_proba"):
        risk_prob = lda.predict_proba(X_all_scaled)[:, 1]
    else:
        risk_pred = lda.predict(X_all_scaled)
        risk_prob = risk_pred.astype(float)

    # --- Compute safe price ---
    safe_price = feature_df["mean_price_usd"].values * (1 - alpha * risk_prob)

    report_df_LDA = pd.DataFrame({
        "market_hash_name": feature_df.index,
        "mean_price_usd": feature_df["mean_price_usd"].values,
        "volatility": feature_df["volatility"].values,
        "spike_count": feature_df["spike_count"].values,
        "risk_score_model": risk_prob,
        "safe_price_usd": safe_price
    })

    report_df_LDA = report_df_LDA.sort_values(by="risk_score_model", ascending=False)
    return report_df_LDA

In [ ]:
lda, scaler, labeled_feature_df = train_risk_model(feature_df)
report_df_LDA = generate_client_report(labeled_feature_df, lda, scaler, alpha=0.25)
print(report_df_LDA.head(10))

In [ ]:
# Ensure 'risky_label' exists before training RF
if "risky_label" not in feature_df.columns:
    spike_threshold = feature_df["spike_count"].quantile(0.9)
    vol_threshold   = feature_df["volatility"].quantile(0.9)
    feature_df["risky_label"] = np.where(
        (feature_df["spike_count"] >= spike_threshold) |
        (feature_df["volatility"]   >= vol_threshold),
        1, 0
    )


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import os, time

X = feature_df[[
    "mean_price_usd",
    "mean_pct_change",
    "volatility",
    "mean_deviation",
    "spike_count"
]].copy()
y = feature_df["risky_label"].values

# Clean data
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = np.clip(X, -1e6, 1e6)
X = np.array(X, dtype=float)

# Train/test split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 2️⃣ Define Custom Random Forest

class MyRandomForest:
    def __init__(self, n_trees=10, sample_ratio=0.3, max_depth=None, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.sample_ratio = sample_ratio
        self.n_features = n_features
        self.trees = []

    def bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, int(n_samples * self.sample_ratio), replace=True)
        return X[indices], y[indices]

    def fit(self, X, y):
        self.trees = []
        n_total_features = X.shape[1]
        for _ in range(self.n_trees):
            # Select random subset of features
            if self.n_features is None:
                selected_features = np.arange(n_total_features)
            else:
                selected_features = np.random.choice(n_total_features, self.n_features, replace=False)
            X_sample, y_sample = self.bootstrap_sample(X[:, selected_features], y)

            tree = DecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(X_sample, y_sample)
            self.trees.append((tree, selected_features))

    def predict(self, X):
        tree_predictions = []
        for tree, features in self.trees:
            preds = tree.predict(X[:, features])
            tree_predictions.append(preds)
        tree_predictions = np.array(tree_predictions).T
        final_predictions = [Counter(row).most_common(1)[0][0] for row in tree_predictions]
        return np.array(final_predictions)



# 3️⃣ Train & Evaluate Model


best_sample = 0.5
best_trees = 15
best_depth = None
num_features = int(x_train.shape[1] * 0.6)

start_time = time.time()
rf_model = MyRandomForest(
    n_trees=best_trees,
    sample_ratio=best_sample,
    max_depth=best_depth,
    n_features=num_features
)
rf_model.fit(x_train, y_train)
training_time = time.time() - start_time

# Evaluate
y_pred = rf_model.predict(x_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("\n🌲 Random Forest Model Evaluation")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Training Time: {training_time:.2f} seconds")


In [ ]:
# ⚡ Fast SVM Model — Risk Classification & Safe Price Estimation

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


# 2️⃣ Ensure risky_label exists

if "risky_label" not in feature_df.columns:
    spike_threshold = feature_df["spike_count"].quantile(0.9)
    vol_threshold   = feature_df["volatility"].quantile(0.9)
    feature_df["risky_label"] = np.where(
        (feature_df["spike_count"] >= spike_threshold) |
        (feature_df["volatility"]   >= vol_threshold),
        1, 0
    )


# 3️⃣ Prepare feature matrix and target

X = feature_df[[
    "mean_price_usd",
    "mean_pct_change",
    "volatility",
    "mean_deviation",
    "spike_count"
]].copy()
y = feature_df["risky_label"].values

# Clean
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = np.clip(X, -1e6, 1e6)
X = np.array(X, dtype=float)

# Split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = MinMaxScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


# 4️⃣ Train Fast SVM (single run, no grid search)

print("🚀 Training Fast SVM (kernel='rbf', C=10, gamma='scale') ...")
svm = SVC(C=10, kernel="rbf", gamma="scale")
svm.fit(x_train_scaled, y_train)
print("✅ Training complete!")


# 5️⃣ Evaluate

y_pred = svm.predict(x_test_scaled)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="coolwarm")
plt.title("SVM Confusion Matrix (Fast Model)")
plt.show()


# 6️⃣ Save Model

os.makedirs("../python_models/models/SVM_models", exist_ok=True)
joblib.dump(svm, "../python_models/models/SVM_models/svm_best_model.pkl")
joblib.dump(scaler, "../python_models/models/SVM_models/svm_scaler.pkl")
print("💾 Saved model → ../python_models/models/SVM_models/svm_best_model.pkl")


# 7️⃣ Predict Risk and Safe Prices

X_scaled = scaler.transform(X)
risk_pred = svm.predict(X_scaled)

feature_df["risk_score_SVM"] = risk_pred
alpha = 0.25
feature_df["safe_price_usd_SVM"] = feature_df["mean_price_usd"] * (1 - alpha * feature_df["risk_score_SVM"])


# 8️⃣ Save Report

os.makedirs("../data/report", exist_ok=True)
svm_report = feature_df[[
    "mean_price_usd", "volatility", "spike_count",
    "risk_score_SVM", "safe_price_usd_SVM"
]].sort_values(by="risk_score_SVM", ascending=False)

svm_report.to_csv("../data/report/svm_safe_prices.csv", index=True)
print("💾 Saved → ../data/report/svm_safe_prices.csv")


# 9️⃣ Visualize Safe vs Market Price

plt.figure(figsize=(10,6))
plt.scatter(svm_report["mean_price_usd"], svm_report["safe_price_usd_SVM"],
            c=svm_report["risk_score_SVM"], cmap="coolwarm", alpha=0.7)
plt.colorbar(label="Risk Prediction (0=Safe, 1=Risky)")
plt.xlabel("Market Price (USD)")
plt.ylabel("Safe Price (USD)")
plt.title("SVM — Safe vs Market Prices (Fast Version)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# 🤖 Logistic Regression — Risk Classification & Safe Price Estimation

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay



# 2️⃣ Ensure risky_label exists

if "risky_label" not in feature_df.columns:
    spike_threshold = feature_df["spike_count"].quantile(0.9)
    vol_threshold   = feature_df["volatility"].quantile(0.9)
    feature_df["risky_label"] = np.where(
        (feature_df["spike_count"] >= spike_threshold) |
        (feature_df["volatility"]   >= vol_threshold),
        1, 0
    )


# 3️⃣ Prepare features and labels

X = feature_df[[
    "mean_price_usd",
    "mean_pct_change",
    "volatility",
    "mean_deviation",
    "spike_count"
]].copy()
y = feature_df["risky_label"].values

# Clean data
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = np.clip(X, -1e6, 1e6)
X = np.array(X, dtype=float)

# Split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = MinMaxScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)


# 4️⃣ Train Logistic Regression

print("🚀 Training Logistic Regression model ...")
log_reg = LogisticRegression(max_iter=1000, C=1.0)
log_reg.fit(x_train_scaled, y_train)
print("✅ Training complete!")


# 5️⃣ Evaluate Model

y_pred = log_reg.predict(x_test_scaled)
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Purples")
plt.title("Logistic Regression Confusion Matrix")
plt.show()


# 7️⃣ Predict Risk and Safe Prices

X_scaled = scaler.transform(X)
risk_prob = log_reg.predict_proba(X_scaled)[:, 1]

feature_df["risk_score_LogReg"] = risk_prob
alpha = 0.25  # same discount rate
feature_df["safe_price_usd_LogReg"] = feature_df["mean_price_usd"] * (1 - alpha * risk_prob)


# 8️⃣ Save Report

os.makedirs("../data/report", exist_ok=True)
logreg_report = feature_df[[
    "mean_price_usd", "volatility", "spike_count",
    "risk_score_LogReg", "safe_price_usd_LogReg"
]].sort_values(by="risk_score_LogReg", ascending=False)

logreg_report.to_csv("../data/report/logistic_safe_prices.csv", index=True)
print("💾 Saved → ../data/report/logistic_safe_prices.csv")


# 9️⃣ Visualize Safe vs Market Price

plt.figure(figsize=(10,6))
plt.scatter(logreg_report["mean_price_usd"], logreg_report["safe_price_usd_LogReg"],
            c=logreg_report["risk_score_LogReg"], cmap="Purples", alpha=0.7)
plt.colorbar(label="Predicted Risk Probability (0–1)")
plt.xlabel("Market Price (USD)")
plt.ylabel("Safe Price (USD)")
plt.title("Logistic Regression — Safe vs Market Prices")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
%pip install seaborn


In [ ]:
# 🧠 Bayesian Gaussian Mixture Model (BGMM)

import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import BayesianGaussianMixture
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


# 2️⃣ Clean and prepare numeric data

X = feature_df[[
    "mean_price_usd",
    "mean_pct_change",
    "volatility",
    "mean_deviation",
    "spike_count"
]].copy()

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
X = np.clip(X, -1e6, 1e6)
X = np.array(X, dtype=float)

# Standardize
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


# 3️⃣ Train Bayesian Gaussian Mixture Model

print("🚀 Training BGMM model (n_components=10)...")
bgmm = BayesianGaussianMixture(
    n_components=10,
    covariance_type='full',
    max_iter=500,
    random_state=42
)
bgmm.fit(X_scaled)
print("✅ BGMM training complete!")


# 4️⃣ Compute anomaly scores and define threshold

scores = -bgmm.score_samples(X_scaled)  # higher score = more anomalous

# dynamic threshold (95th percentile)
threshold = np.percentile(scores, 95)
anomaly_flag = (scores > threshold).astype(int)
print(f"[INFO] Threshold (95th percentile): {threshold:.4f}")

# 5️⃣ Evaluate anomaly spread (optional)

plt.figure(figsize=(8, 5))
sns.histplot(scores, bins=50, kde=True, color="skyblue")
plt.axvline(threshold, color="red", linestyle="--", label="95th Percentile Threshold")
plt.title("BGMM Anomaly Score Distribution")
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


# 6️⃣ Integrate anomaly results with features

feature_df["risk_score_BGMM"] = (scores - scores.min()) / (scores.max() - scores.min())
feature_df["anomaly_flag"] = anomaly_flag

# Safe price = lower bound based on risk probability
alpha = 0.25
feature_df["safe_price_usd_BGMM"] = feature_df["mean_price_usd"] * (1 - alpha * feature_df["risk_score_BGMM"])


# 7️⃣ Save model, scaler, and report

os.makedirs("../python_models/models/BGMM_model", exist_ok=True)
os.makedirs("../data/report", exist_ok=True)

joblib.dump(bgmm, "../python_models/models/BGMM_model/bgmm_model.pkl")
joblib.dump(scaler, "../python_models/models/BGMM_model/bgmm_scaler.pkl")
joblib.dump(threshold, "../python_models/models/BGMM_model/bgmm_threshold.pkl")
print("💾 Saved model and scaler → ../python_models/models/BGMM_model/")

bgmm_report = feature_df[[
    "mean_price_usd", "volatility", "spike_count",
    "risk_score_BGMM", "safe_price_usd_BGMM"
]].sort_values(by="risk_score_BGMM", ascending=False)

bgmm_report.to_csv("../data/report/bgmm_safe_prices.csv", index=True)
print("💾 Saved → ../data/report/bgmm_safe_prices.csv")


# 8️⃣ Visualize Safe vs Market Prices

plt.figure(figsize=(10,6))
plt.scatter(bgmm_report["mean_price_usd"], bgmm_report["safe_price_usd_BGMM"],
            c=bgmm_report["risk_score_BGMM"], cmap="coolwarm", alpha=0.7)
plt.colorbar(label="Risk Score (BGMM)")
plt.xlabel("Market Price (USD)")
plt.ylabel("Safe Price (USD)")
plt.title("BGMM — Safe vs Market Prices (Unsupervised Anomaly Model)")
plt.grid(True)
plt.tight_layout()
plt.show()
